# ⛏️ Optimización de recuperación de oro

Modelo predictivo para estimar la recuperación de oro en las etapas
rougher y final. Se evalúa el desempeño con la métrica sMAPE.

## 0. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import cross_val_score, KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer

plt.style.use('default')


## 1. Carga y exploración inicial de los datos

In [ ]:
TRAIN_PATH = 'data/gold_recovery_train.csv'
TEST_PATH = 'data/gold_recovery_test.csv'
FULL_PATH = 'data/gold_recovery_full.csv'

train = pd.read_csv(TRAIN_PATH, index_col='date', parse_dates=True)
test = pd.read_csv(TEST_PATH, index_col='date', parse_dates=True)
full = pd.read_csv(FULL_PATH, index_col='date', parse_dates=True)

print('Dimensiones:')
print('train:', train.shape)
print('test :', test.shape)
print('full :', full.shape)

display(train.head())
display(test.head())
display(full.head())

print('\nInformación del conjunto de entrenamiento:')
train.info()


## 1.2 Comprobar cálculo de la recuperación

In [ ]:
# Fórmula:
# recovery = (C * (F - T)) / (F * (C - T)) * 100
def calc_recovery(row):
    c = row['rougher.output.concentrate_au']
    f = row['rougher.input.feed_au']
    t = row['rougher.output.tail_au']

    if pd.isna(c) or pd.isna(f) or pd.isna(t):
        return np.nan

    den = f * (c - t)
    if den == 0 or np.isclose(den, 0):
        return np.nan

    val = (c * (f - t)) / den * 100
    if not np.isfinite(val):
        return np.nan
    return val

train['recovery_calc'] = train.apply(calc_recovery, axis=1)
mask = train['rougher.output.recovery'].notna() & train['recovery_calc'].notna()
y_true = train.loc[mask, 'rougher.output.recovery']
y_pred = train.loc[mask, 'recovery_calc']
eam = (y_true - y_pred).abs().mean()
print('\nEAM entre recuperación real y calculada:', eam)


## 1.3 Columnas ausentes en el conjunto de prueba

In [ ]:
target_cols = ['rougher.output.recovery', 'final.output.recovery']
train_features_all = train.drop(columns=target_cols, errors='ignore')
test_features_all = test.copy()

missing_in_test = train_features_all.columns.difference(test_features_all.columns)
print('\nColumnas presentes en train pero NO en test:')
print(missing_in_test)

print('\nTipos de datos de esas columnas:')
train[missing_in_test].info()


## 1.4 Preprocesamiento de datos

In [ ]:
# 1) Usar solo columnas comunes entre train y test
common_cols = test_features_all.columns
train_features = train_features_all[common_cols].copy()
test_features = test_features_all[common_cols].copy()

print('\nDimensiones después de usar solo columnas comunes:')
print('train_features:', train_features.shape)
print('test_features :', test_features.shape)

# 2) Imputación de NaN con la mediana
train_features = train_features.fillna(train_features.median())
test_features = test_features.fillna(test_features.median())

# 3) Objetivos (pueden contener NaN)
y_train = train[target_cols].copy()
print('\nDimensiones de y_train (targets):', y_train.shape)


## 2. Análisis exploratorio de datos (EDA)

In [ ]:
metals = ['au', 'ag', 'pb']
for metal in metals:
    plt.figure(figsize=(8, 5))
    col_feed = f'rougher.input.feed_{metal}'
    if col_feed in train.columns:
        train[col_feed].hist(alpha=0.5, bins=50, label='rougher.input')
    col_rougher = f'rougher.output.concentrate_{metal}'
    if col_rougher in train.columns:
        train[col_rougher].hist(alpha=0.5, bins=50, label='rougher.output')
    col_final = f'final.output.concentrate_{metal}'
    if col_final in train.columns:
        train[col_final].hist(alpha=0.5, bins=50, label='final.output')
    plt.title(f'Distribución de concentración de {metal.upper()} por etapa')
    plt.xlabel('Concentración')
    plt.ylabel('Frecuencia')
    plt.legend()
    plt.show()

size_col = 'rougher.input.feed_size'
if size_col in train.columns and size_col in test.columns:
    plt.figure(figsize=(8, 5))
    train[size_col].hist(alpha=0.5, bins=50, label='train')
    test[size_col].hist(alpha=0.5, bins=50, label='test')
    plt.title('Distribución del tamaño de partículas (rougher.input.feed_size)')
    plt.xlabel('Tamaño de partícula')
    plt.ylabel('Frecuencia')
    plt.legend()
    plt.show()
else:
    print(f'\nNo se encontró la columna {size_col} en ambos conjuntos.')

def total_concentration(df, stage_prefix):
    cols = [c for c in df.columns if c.startswith(stage_prefix) and (c.endswith('_au') or c.endswith('_ag') or c.endswith('_pb') or c.endswith('_sol'))]
    if len(cols) == 0:
        return pd.Series(dtype=float)
    return df[cols].sum(axis=1)

bad_idx_all = set()
for stage in ['rougher.input', 'rougher.output', 'final.output']:
    total = total_concentration(train, stage)
    if total.empty:
        continue
    plt.figure(figsize=(8, 5))
    total.hist(bins=50)
    plt.title(f'Total de concentraciones en {stage}')
    plt.xlabel('Suma de concentraciones')
    plt.ylabel('Frecuencia')
    plt.show()
    bad_idx_stage = total[(total < 0) | (total > 100)].index
    print(f'Etapa {stage} - número de valores anómalos:', len(bad_idx_stage))
    bad_idx_all |= set(bad_idx_stage)

print('\nNúmero total de índices anómalos a eliminar:', len(bad_idx_all))
train = train.drop(index=bad_idx_all)
y_train = y_train.drop(index=bad_idx_all)
train_features = train_features.drop(index=bad_idx_all)
full = full.drop(index=bad_idx_all, errors='ignore')

print('Dimensiones después de eliminar anomalías:')
print('train:', train.shape)
print('train_features:', train_features.shape)
print('y_train:', y_train.shape)
print('full:', full.shape)


## 3. Limpieza final antes de entrenar el modelo

In [ ]:
mask_targets = y_train.notna().all(axis=1)
mask_features = (np.isfinite(train_features).all(axis=1) & train_features.notna().all(axis=1))
mask_good = mask_targets & mask_features

X_train = train_features.loc[mask_good].copy()
y_train_targets = y_train.loc[mask_good].copy()

print('\nDimensiones finales para entrenamiento:')
print('X_train:', X_train.shape)
print('y_train_targets:', y_train_targets.shape)


## 4. Función sMAPE y métrica final

In [ ]:
def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(np.abs(y_pred[mask] - y_true[mask]) / denominator[mask]) * 100

def final_smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_true_rougher = y_true[:, 0]
    y_true_final = y_true[:, 1]
    y_pred_rougher = y_pred[:, 0]
    y_pred_final = y_pred[:, 1]
    return 0.25 * smape(y_true_rougher, y_pred_rougher) + 0.75 * smape(y_true_final, y_pred_final)

def final_smape_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    return -final_smape(y.values, y_pred)

smape_scorer = make_scorer(final_smape, greater_is_better=False)


## 5. Entrenamiento y validación cruzada

In [ ]:
models = {
    'LinearRegression': MultiOutputRegressor(LinearRegression()),
    'RandomForest': MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=200,
            max_depth=6,
            random_state=42,
            n_jobs=-1
        )
    )
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
print('\nResultados de validación cruzada (sMAPE medio):')
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train_targets, cv=cv, scoring=smape_scorer)
    print(f'{name}: {-scores.mean():.4f}')


## 6. Entrenar el mejor modelo y evaluar en test

In [ ]:
best_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=6,
        random_state=42,
        n_jobs=-1
    )
)
best_model.fit(X_train, y_train_targets)

X_test = test_features.copy()
y_test = full.loc[X_test.index, target_cols]

mask_test = y_test.notna().all(axis=1)
X_test_clean = X_test.loc[mask_test]
y_test_clean = y_test.loc[mask_test]

y_pred_test = best_model.predict(X_test_clean)
test_metric = final_smape(y_test_clean.values, y_pred_test)
print('\nMétrica final en el conjunto de prueba (sMAPE global):', test_metric)
